In [3]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()  # project root if notebook is in notebooks/
COURSES_PATH = ROOT / "data/structured/courses.json"
CUTOFFS_PATH = ROOT / "data/structured/cutoffs.json"
DB_PATH = ROOT / "data/structured/unipath.db"

sns.set_theme(style="whitegrid")
%matplotlib inline

# courses.json — shape & structure

Top-level: **JSON array** of course objects (one per UGC handbook catalogue entry).

```
courses.json
└── [ course, course, ... ]     ← 101 items
    └── {
          section,               ← handbook section id (string)
          course_name,           ← display name (string)
          course_code_base,      ← 3-digit UGC code (string)
          proposed_intake,       ← intake count (string, may have commas)
          eligibility,           ← A/L rules text (string | null)
          degree_programme,      ← degree title (string | null)
          available_universities,← list of uni names (string[])
          duration,              ← e.g. "04 years" (string | null)
          medium                 ← language of instruction (string | null)
        }
```

In [ ]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd

courses_raw = json.loads(COURSES_PATH.read_text(encoding="utf-8"))

# --- Top-level shape ---
print("=" * 60)
print("TOP-LEVEL SHAPE")
print("=" * 60)
print(f"Type:        {type(courses_raw).__name__}")
print(f"Length:      {len(courses_raw)} courses")
print(f"File size:   {COURSES_PATH.stat().st_size / 1024:.1f} KB")

# --- Field schema (keys on every record) ---
all_keys = sorted({k for c in courses_raw for k in c})
print(f"\nFields ({len(all_keys)}): {all_keys}")

# --- Per-field type & null stats ---
rows = []
for field in all_keys:
    values = [c.get(field) for c in courses_raw]
    non_null = [v for v in values if v is not None]
    types = Counter(type(v).__name__ for v in non_null)
    empty_list = sum(1 for v in values if v == [])
    rows.append({
        "field": field,
        "non_null": len(non_null),
        "null": len(values) - len(non_null),
        "empty_list": empty_list if field == "available_universities" else None,
        "types": dict(types),
        "example": non_null[0] if non_null else None,
    })

schema_df = pd.DataFrame(rows).set_index("field")
display(schema_df)

In [ ]:
# --- Nested structure: available_universities ---
uni_counts = [len(c.get("available_universities") or []) for c in courses_raw]

print("available_universities (nested list)")
print(f"  min universities: {min(uni_counts)}")
print(f"  max universities: {max(uni_counts)}")
print(f"  mean:             {sum(uni_counts)/len(uni_counts):.1f}")
print(f"  empty lists:      {sum(1 for n in uni_counts if n == 0)}")

# Flatten to long format (course × university rows)
uni_rows = []
for c in courses_raw:
    for uni in c.get("available_universities") or []:
        uni_rows.append({
            "course_code": c["course_code_base"],
            "course_name": c["course_name"],
            "university": uni,
        })
uni_long = pd.DataFrame(uni_rows)
print(f"\nFlattened course×university rows: {len(uni_long)}")
print(f"Unique universities: {uni_long['university'].nunique() if len(uni_long) else 0}")

display(uni_long.head(10))

In [ ]:
# --- Example records: incomplete vs complete ---

def find_course(name: str) -> dict:
    for c in courses_raw:
        if c["course_name"] == name:
            return c
    raise KeyError(name)

print("INCOMPLETE example — Arts (019)")
print("-" * 40)
pprint(find_course("Arts"))

print("\n\nCOMPLETE example — Computer Science (012)")
print("-" * 40)
pprint(find_course("Computer Science"))

In [ ]:
# --- Section grouping (handbook hierarchy) ---

sections = pd.DataFrame([
    {
        "section": c["section"],
        "code": c["course_code_base"],
        "name": c["course_name"],
        "uni_count": len(c.get("available_universities") or []),
    }
    for c in courses_raw
]).sort_values("section")

print(f"Unique section prefixes: {sections['section'].str.rsplit('.', n=1).str[0].nunique()}")
display(sections.groupby(sections["section"].str.extract(r'^(2\.2\.\d+)')[0]).agg(
    courses=("name", "count"),
    with_unis=("uni_count", lambda s: (s > 0).sum()),
).rename_axis("handbook_group"))

sections.head(20)

In [4]:
courses_raw = json.loads(COURSES_PATH.read_text(encoding="utf-8"))

courses = pd.DataFrame([
    {
        "code": c["course_code_base"],
        "name": c["course_name"],
        "section": c.get("section"),
        "intake": pd.to_numeric(str(c.get("proposed_intake", "")).replace(",", ""), errors="coerce"),
        "uni_count": len(c.get("available_universities") or []),
        "has_eligibility": c.get("eligibility") is not None,
        "has_degree": c.get("degree_programme") is not None,
        "universities": ", ".join(c.get("available_universities") or []),
    }
    for c in courses_raw
])

courses["complete"] = courses["uni_count"] > 0 & courses["has_eligibility"]
courses.sort_values("intake", ascending=False).head(15)

,code,name,section,intake,uni_count,has_eligibility,has_degree,universities,complete
0,019,Arts,2.2.1.1,6983,0,False,False,,False
1,016,Management,2.2.2.1,5424,0,False,False,,False
44,013,Physical Science,2.2.4.5,2262,8,True,True,"University of Colombo, University of Peradeniy...",True
40,008,Engineering,2.2.4.1,2238,0,True,False,,False
14,006,Biological Science,2.2.3.11,1683,8,True,True,"University of Colombo, University of Peradeniy...",True
52,102,Engineering Technology (ET),2.2.5.1,1336,0,True,False,,False
53,103,Biosystems Technology (BST),2.2.6.1,1324,0,True,False,,False
4,018,Commerce,2.2.2.4,884,0,False,False,,False
46,015,Applied Sciences (Physical Science),2.2.4.7,778,0,True,False,,False
45,012,Computer Science,2.2.4.6,686,4,True,True,University of Colombo School of Computing (UCS...,True
